In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import skimage.io
import keras.backend as K
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau,ModelCheckpoint,EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Input, Dropout, Flatten, Conv2D


In [2]:
train_datagen = ImageDataGenerator(#rotation_range = 180,
                                         width_shift_range = 0.1,
                                         height_shift_range = 0.1,
                                         horizontal_flip = True,
                                         rescale = 1./255,
                                         #zoom_range = 0.2,
                                         validation_split = 0.2
                                        )
valid_datagen = ImageDataGenerator(rescale = 1./255,
                                         validation_split = 0.2)
test_datagen = ImageDataGenerator(rescale = 1./255,
                                         validation_split = 0.2)


In [3]:
train_dataset=train_datagen.flow_from_directory(directory='data_small/train',
                                               target_size=(48,48),
                                               class_mode='categorical',
                                               subset='training',
                                               batch_size=64)


Found 560 images belonging to 7 classes.


In [4]:
valid_dataset=valid_datagen.flow_from_directory(directory='data_small/test',
                                               target_size=(48,48),
                                               class_mode='categorical',
                                               batch_size=64)


Found 700 images belonging to 7 classes.


In [5]:
test_dataset=test_datagen.flow_from_directory(directory='data_small/test',
                                               target_size=(48,48),
                                               class_mode='categorical',
                                               batch_size=64)


Found 700 images belonging to 7 classes.


In [6]:
train_datagen = ImageDataGenerator(
    # Other parameters...
    rotation_range=20,  # Example: Add rotation for augmentation
    zoom_range=0.2  # Example: Add zooming for augmentation
)

In [7]:
# the fxies here are for reproducing purposes
from tensorflow.keras.layers import BatchNormalization, Activation, MaxPooling2D
from tensorflow.keras.layers import SeparableConv2D
from tensorflow.keras.layers import concatenate
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras import Model
from tensorflow.keras.layers import LeakyReLU  
from tensorflow.keras.regularizers import l2 

inputs=Input((64,64,3))

h=Conv2D(64,(1,1),padding='same',activation='relu')(inputs)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=Conv2D(64,(3,3))(h)
h=BatchNormalization()(h)
#     h=MaxPooling2D((2,2),strides=(2,2))(h)
h=Activation('relu')(h)

b=Conv2D(128,(1,1),strides=(2,2))(h)
b=BatchNormalization()(b)

h=SeparableConv2D(128,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=Activation('relu')(h)
h=SeparableConv2D(128,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h=MaxPooling2D((2,2),strides=(2,2))(h)

h=concatenate([h,b],name='first')

b=Conv2D(128,(2,2),strides=(2,2))(h)
b=BatchNormalization()(b)

h=SeparableConv2D(128,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(128,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h=MaxPooling2D((2,2),strides=(2,2))(h)

h=concatenate([h,b],name='second')

b=Conv2D(256,(1,1),padding='same')(h)
b=BatchNormalization()(b)
b=MaxPooling2D((2,2),strides=(2,2))(b)

h=SeparableConv2D(256,(3,3),padding='same')(h)
h=BatchNormalization()(h)
# h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=Activation('relu')(h)
h=SeparableConv2D(256,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h=MaxPooling2D((2,2),strides=(2,2))(h)

h=concatenate([h,b],name='third')
b=h

h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)

h=concatenate([h,b],name='fourth')

b=Conv2D(512,(1,1),padding='same')(h)
b=BatchNormalization()(b)
b=MaxPooling2D((2,2),strides=(2,2))(b)

h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h=MaxPooling2D((2,2),strides=(2,2))(h)

h=concatenate([h,b],name='fifth')

b=Conv2D(1024,(1,1),padding='same')(h)
b=BatchNormalization()(b)
b=MaxPooling2D((2,2),strides=(2,2))(b)

h=SeparableConv2D(1024,(3,3),padding='same')(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(1024,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h=MaxPooling2D((2,2),strides=(2,2))(h)

h=concatenate([h,b],name='sixth')
b=h

h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)

h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(256,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(128,(3,3),padding='same')(h)
h=BatchNormalization()(h)

h=concatenate([h,b],name='seventh')
b=h

b=Conv2D(256,(1,1),strides=(1,1))(h)
b=BatchNormalization()(b)

h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(1024,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)
h=SeparableConv2D(512,(3,3),padding='same')(h)
h=BatchNormalization()(h)

h=concatenate([h,b],name='eighth')

h=SeparableConv2D(256,(3,3),padding='same')(h)
h=BatchNormalization()(h)
h = Activation('relu')(h)   # fix # h=tf.nn.relu(h)



x = GlobalAveragePooling2D()(h)

# Fully Connected Layer
x = Dense(1024)(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.4)(x)

x = Dense(512)(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.4)(x)

# Additional Fully Connected Layer
x = Dense(256)(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.3)(x)

x = Dense(128)(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.2)(x)


# Output Layer
outputs = Dense(7, activation='softmax')(x)



# Create the model
model = Model(inputs=inputs, outputs=outputs)
model.summary()


/usr/local/lib/python3.10/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 64, 64, 3)      │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 64, 64, 64)     │            256 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 64, 64, 64)     │            256 │ conv2d[0][0]           │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 64, 64, 64)     │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_1 (Conv2D)         │ (None, 62, 62, 64)     │         36,928 │ activation[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 62, 62, 64)     │            256 │ conv2d_1[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_1 (Activation) │ (None, 62, 62, 64)     │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv2d          │ (None, 62, 62, 128)    │          8,896 │ activation_1[0][0]     │
│ (SeparableConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_3     │ (None, 62, 62, 128)    │            512 │ separable_conv2d[0][0] │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_2 (Activation) │ (None, 62, 62, 128)    │              0 │ batch_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_3 (Activation) │ (None, 62, 62, 128)    │              0 │ activation_2[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv2d_1        │ (None, 62, 62, 128)    │         17,664 │ activation_3[0][0]     │
│ (SeparableConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_4     │ (None, 62, 62, 128)    │            512 │ separable_conv2d_1[0]… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_2 (Conv2D)         │ (None, 31, 31, 128)    │          8,320 │ activation_1[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d             │ (None, 31, 31, 128)    │              0 │ batch_normalization_4… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_2     │ (None, 31, 31, 128)    │            512 │ conv2d_2[0][0]         │
│ (BatchNormalization) 

 Total params: 11,535,495 (44.00 MB)

 Trainable params: 11,512,199 (43.92 MB)

 Non-trainable params: 23,296 (91.00 KB)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', np.unique(train_dataset.labels), train_dataset.labels)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Compile the model with class weights
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=METRICS, class_weight=class_weight_dict)


TypeError: compute_class_weight() takes 1 positional argument but 3 were given

In [ ]:
from tensorflow.keras.utils import plot_model
from IPython.display import Image
plot_model(model,to_file='model.png',show_shapes=True,show_layer_names=True)
Image(filename='model.png')


In [8]:
def f1_score(y_true,y_pred):
    true_positives=K.sum(K.round(K.clip(y_true*y_pred,0,1)))
    possible_positives=K.sum(K.round(K.clip(y_true,0,1)))
    predicted_positives=K.sum(K.round(K.clip(y_pred,0,1)))
    precision=true_positives/(predicted_positives+K.epsilon())
    recall=true_positives/(possible_positives+K.epsilon())
    f1_val=2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val


In [9]:
METRICS=[
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc'),
      f1_score,
]


In [ ]:
lrd=ReduceLROnPlateau(monitor='val_loss',patience=20,verbose=1,factor=0.50,min_lr=0.00005)
mcp=ModelCheckpoint('model.h5')
es=EarlyStopping(verbose=1,patience=20)


In [ ]:
model.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=METRICS)


In [ ]:
history=model.fit(train_dataset,validation_data=valid_dataset,epochs=20,verbose=1,callbacks=[lrd,mcp,es])


In [ ]:
model_acc=model.evaluate(test_dataset,verbose=0)[1]
preds=model.predict(test_dataset)
y_preds=np.argmax(preds,axis=1)
y_test=np.array(test_dataset.labels)


In [ ]:
train_dir='../input/fer2013/train'
test_dir='../input/fer2013/test'

class_labels=['Angry','Disgust','Fear','Happy','Neutral','Sad','Surprise']


In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
cm_data=confusion_matrix(y_test,y_preds)
cm=pd.DataFrame(cm_data,columns=class_labels,index=class_labels)
cm.index.name="Actual"
cm.columns.name="Predicted"
plt.figure(figsize=(20,10))
plt.title('Confusion Matrix',fontsize=20)
sn.set(font_scale=1.2)
sn.set(font_scale=1.2)
ax=sn.heatmap(cm,cbar=False,cmap="Blues",annot=True,annot_kws={"size":16},fmt='g')


In [ ]:
from sklearn.preprocessing import LabelBinarizer
fig,c_ax=plt.subplots(1,1,figsize=(15,8))
from sklearn.metrics import roc_curve,auc,roc_auc_score

def multiclass_roc_auc_score(y_test,y_preds,average='macro'):
    lb=LabelBinarizer()
    lb.fit(y_test)
    y_test=lb.transform(y_test)
    for(idx,c_label) in enumerate(class_labels):
        fpr,tpr,thresholds=roc_curve(y_test[:,idx].astype(int),y_preds[:,idx])
        c_ax.plot(fpr,tpr,lw=2,label='%s (AUC:%0.2f)'%(c_label,auc(fpr,tpr)))
    c_ax.plot(fpr,fpr,'black',linestyle='dashed',lw=4,label='Random Guessing')
    return roc_auc_score(y_test,y_preds,average=average)

print('ROC AUC score:',multiclass_roc_auc_score(y_test,preds,average='micro'))
plt.xlabel('FALSE POSITIVE RATE',fontsize=18)
plt.ylabel('TRUE POSITIVE RATE',fontsize=16)
plt.legend(fontsize=11.5)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Assuming you have 'history' object with 'loss', 'val_loss', 'accuracy', and 'val_accuracy' data

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot training and validation loss
ax1.plot(history.history['loss'])
ax1.plot(history.history['val_loss'])
ax1.set_title('Model Loss')
ax1.set_ylabel('Loss')
ax1.set_xlabel('Epoch')
ax1.legend(['Train', 'Validation'], loc='upper right')

# Plot training and validation accuracy
ax2.plot(history.history['accuracy'])
ax2.plot(history.history['val_accuracy'])
ax2.set_title('Model Accuracy')
ax2.set_ylabel('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend(['Train', 'Validation'], loc='lower right')

# Adjust the layout to prevent overlap
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the saved model
model = load_model('model.h5', custom_objects={'f1_score': f1_score})
# Define a function to predict the emotion
def predict_emotion(image_path):
    # Load and preprocess the input image
    img = image.load_img(image_path, target_size=(64, 64))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    img = img / 255.0  # Normalize the pixel values

    # Make predictions
    predictions = model.predict(img)

    # Map predicted class to emotion label (modify this mapping based on your dataset)
    emotion_labels = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
    predicted_emotion = emotion_labels[np.argmax(predictions)]

    return predicted_emotion

# Path to the image you want to test
image_path = '/kaggle/input/fer2013/test/surprise/PrivateTest_10072988.jpg'

# Get the predicted emotion
predicted_emotion = predict_emotion(image_path)

print("Predicted Emotion:", predicted_emotion)

In [ ]:
from sklearn.metrics import confusion_matrix

# Your code for creating the confusion matrix
cm_data = confusion_matrix(y_test, y_preds)

# Calculate confusion matrix accuracy (True Positive Rate) for each class
class_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
accuracies = {}

for i, label in enumerate(class_labels):
    TP = cm_data[i, i]  # True Positives for the current class
    FN = np.sum(cm_data[i, :]) - TP  # False Negatives for the current class
    accuracy = TP / (TP + FN)  # Confusion matrix accuracy (True Positive Rate)
    accuracies[label] = accuracy

# Print the confusion matrix accuracy for each class
for label, accuracy in accuracies.items():
    print(f'Accuracy for class {label}: {accuracy:.2f}')

# VGG

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import skimage.io
import keras.backend as K
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense,Flatten,Dropout,BatchNormalization,Activation
from tensorflow.keras.models import Model,Sequential
from keras.applications.nasnet import NASNetLarge
from tensorflow.keras.callbacks import ReduceLROnPlateau,ModelCheckpoint,EarlyStopping
from tensorflow.keras.optimizers import Adam

In [ ]:
train_datagen=ImageDataGenerator(rescale=1./255,validation_split=0.2,
                                rotation_range=5,
                                width_shift_range=0.2,
                                height_shift_range=0.2,
                                shear_range=0.2,
                                #zoom_range=0.2,
                                horizontal_flip=True,
                                vertical_flip=True,
                                fill_mode='nearest')
valid_datagen=ImageDataGenerator(rescale=1./255,validation_split=0.2)
test_datagen=ImageDataGenerator(rescale=1./255)

In [ ]:
train_dataset=train_datagen.flow_from_directory(directory='../input/fer2013/train',
                                               target_size=(48,48),
                                               class_mode='categorical',
                                               subset='training',
                                               batch_size=64)


In [ ]:
valid_dataset=valid_datagen.flow_from_directory(directory='../input/fer2013/test',
                                               target_size=(48,48),
                                               class_mode='categorical',
                                               batch_size=64)

In [ ]:
test_dataset=test_datagen.flow_from_directory(directory='../input/fer2013/test',
                                             target_size=(48,48),
                                             batch_size=64,
                                             class_mode='categorical')

In [ ]:
base_model=tf.keras.applications.VGG16(input_shape=(48,48,3),include_top=False,weights='imagenet')


In [ ]:
for layer in base_model.layers[:-4]:
    layer.trainable=False

In [ ]:
model=Sequential()
model.add(base_model)
model.add(Dropout(0.5))
model.add(Flatten())
model.add(BatchNormalization())
model.add(Dense(32,kernel_initializer='he_uniform'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(32,kernel_initializer='he_uniform'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(32,kernel_initializer='he_uniform'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dense(7,activation='softmax'))

vgg_model=model
vgg_model.summary()

In [ ]:
from tensorflow.keras.utils import plot_model
from IPython.display import Image
plot_model(model,to_file='vgg.png',show_shapes=True,show_layer_names=True)
Image(filename='vgg.png')

In [ ]:
def f1_score(y_true,y_pred): #taken from old keras source code
    true_positives=K.sum(K.round(K.clip(y_true*y_pred,0,1)))
    possible_positives=K.sum(K.round(K.clip(y_true,0,1)))
    predicted_positives=K.sum(K.round(K.clip(y_pred,0,1)))
    precision=true_positives/(predicted_positives+K.epsilon())
    recall=true_positives/(possible_positives+K.epsilon())
    f1_val=2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val

In [ ]:
METRICS=[
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc'),
      f1_score,
]


In [ ]:
lrd=ReduceLROnPlateau(monitor='val_loss',patience=20,verbose=1,factor=0.50,min_lr=0.00005)
mcp=ModelCheckpoint('vgg_model.h5')
es=EarlyStopping(verbose=1,patience=20)

In [ ]:
model.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=METRICS)


In [ ]:
vgg_history=vgg_model.fit(train_dataset,validation_data=valid_dataset,epochs=10,verbose=1,callbacks=[lrd,mcp,es])

In [ ]:
vgg_acc=vgg_model.evaluate(test_dataset,verbose=0)[1]
preds=vgg_model.predict(test_dataset)
y_preds=np.argmax(preds,axis=1)
y_test=np.array(test_dataset.labels)

In [ ]:
train_dir='../input/fer2013/train'
test_dir='../input/fer2013/test'

class_labels=['Angry','Disgust','Fear','Happy','Neutral','Sad','Surprise']

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
cm_data_vgg=confusion_matrix(y_test,y_preds)
cm_vgg=pd.DataFrame(cm_data_vgg,columns=class_labels,index=class_labels)
cm_vgg.index.name="Actual"
cm_vgg.columns.name="Predicted"
plt.figure(figsize=(20,10))
plt.title('Confusion Matrix',fontsize=20)
sn.set(font_scale=1.2)
sn.set(font_scale=1.2)
ax_vgg=sn.heatmap(cm_vgg,cbar=False,cmap="Blues",annot=True,annot_kws={"size":16},fmt='g')


In [ ]:
from sklearn.preprocessing import LabelBinarizer
fig,c_ax=plt.subplots(1,1,figsize=(15,8))
from sklearn.metrics import roc_curve,auc,roc_auc_score

def multiclass_roc_auc_score1(y_test,y_preds,average="macro"):
    lb=LabelBinarizer()
    lb.fit(y_test)
    y_test=lb.transform(y_test)
    for(idx,c_label) in enumerate(class_labels):
        fpr,tpr,thresholds=roc_curve(y_test[:,idx].astype(int),y_preds[:,idx])
        c_ax.plot(fpr,tpr,lw=2,label='%s (AUC:%0.2f)'%(c_label,auc(fpr,tpr)))
    c_ax.plot(fpr,fpr,'black',linestyle='dashed',lw=4,label='Random Guessing')
    return roc_auc_score(y_test,y_preds,average=average)

print('ROC AUC score:',multiclass_roc_auc_score1(y_test,preds,average='micro'))
plt.xlabel('FALSE POSITIVE RATE',fontsize=18)
plt.ylabel('TRUE POSITIVE RATE',fontsize=16)
plt.legend(fontsize=11.5)
plt.show()

# ResNet50

In [ ]:
base_model1=tf.keras.applications.ResNet50(input_shape=(48,48,3),include_top=False,weights="imagenet")

In [ ]:
for layer in base_model1.layers[:4]:
    layer.trainable=False

In [ ]:
model_resnet=Sequential()
model_resnet.add(base_model1)
model_resnet.add(Dropout(0.5))
model_resnet.add(Flatten())
model_resnet.add(BatchNormalization())
model_resnet.add(Dense(32,kernel_initializer='he_uniform'))
model_resnet.add(BatchNormalization())
model_resnet.add(Activation('relu'))
model_resnet.add(Dropout(0.5))
model_resnet.add(Dense(32,kernel_initializer='he_uniform'))
model_resnet.add(BatchNormalization())
model_resnet.add(Activation('relu'))
model_resnet.add(Dropout(0.5))
model_resnet.add(Dense(32,kernel_initializer='he_uniform'))
model_resnet.add(BatchNormalization())
model_resnet.add(Activation('relu'))
model_resnet.add(Dense(7,activation='softmax'))

#model summary
model_resnet.summary()

In [ ]:
plot_model(model_resnet,to_file='model_resnet.png',show_shapes=True,show_layer_names=True)
Image(filename='model_resnet.png')

In [ ]:
mcp1=ModelCheckpoint('model_resnet.h5')

In [ ]:
model_resnet.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=METRICS)

In [ ]:
history_resnet=model_resnet.fit(train_dataset,validation_data=valid_dataset,epochs=20,verbose=1,callbacks=[lrd,mcp1,es])